In [ ]:
import pandas as pd
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split

# Use the shared loader and the sibling DOHA repository on every platform.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.data_loader import load_cases

df = load_cases().dropna(subset=["full_text", "outcome"])

# Split 20% for testing (reproducible seed 42, stratified)
train_val_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['outcome']
)

# Split the remaining 80% into clean training and validation sets
train_df, val_df = train_test_split(
    train_val_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_val_df['outcome']
)

# These reproduce the team's row-stratified 64/16/20 split.
# Related cases / duplicate documents may cross partitions: do not use these
# exploratory splits as final model evaluation without a leakage review.
output_dir = REPO_ROOT / "data" / "processed" / "splits"
output_dir.mkdir(parents=True, exist_ok=True)
train_df.to_parquet(output_dir / "train.parquet", index=False)
val_df.to_parquet(output_dir / "val.parquet", index=False)
test_df.to_parquet(output_dir / "test.parquet", index=False)

print(f"Exploratory data splits saved to {output_dir}")
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")
